# Tuning v2 — GridSearchCV restringido a `year <= 2019` (sin fuga temporal)

## Mejora respecto a v1
En [`tuning/v1/tune_rf.ipynb`](../v1/tune_rf.ipynb), el `GridSearchCV` se entrenaba con
el dataset **completo** (2001-2024). Como los folds de `GroupKFold` se arman por
**bloque espacial** (no por año), algunas filas de 2020+ terminaban como datos de
**entrenamiento** en ciertos folds durante la búsqueda de hiperparámetros — es decir,
la elección de hiperparámetros pudo verse influenciada indirectamente por patrones de
los años que se suponía iban a quedar reservados como hold-out temporal.

**v2 corrige esto:** el `GridSearchCV` corre únicamente sobre `year <= 2019`. Los años
`>= 2020` quedan completamente fuera de todo el proceso de tuning y solo se usan **una
vez, al final**, como hold-out temporal genuino. La grilla de hiperparámetros (valores
candidatos) es idéntica a v1 — el único cambio es el recorte de datos usado para
buscarlos, para poder aislar el efecto de esta única mejora.

## Qué hace este notebook
1. Carga el mismo dataset congelado (`data/model_dataset/model_dataset.csv`).
2. Corre `GridSearchCV` (10-fold spatial-block CV, PR-AUC) para LR y RF, **solo con
   `year <= 2019`**.
3. Evalúa los modelos afinados con el mismo protocolo de siempre (spatial block CV de
   10 folds sobre el dataset completo + hold-out temporal `>=2020`), agregando también
   **matriz de confusión** (nuevo en v2).
4. Guarda todas las métricas en [`results_v2.csv`](results_v2.csv), en esta misma
   carpeta.


In [1]:
# === Tuning v2 — Setup ===
# Same frozen dataset used by v1 and the baseline models, so results stay comparable.
import os
import glob
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score,
                              confusion_matrix)
from sklearn.base import clone

fixed_path = '../../data/model_dataset/model_dataset.csv'
matches = [fixed_path] if os.path.exists(fixed_path) else glob.glob('../../**/model_dataset.csv', recursive=True)
if not matches:
    raise FileNotFoundError("model_dataset.csv not found — run baseline_comparison first")
print("Loading:", matches[0])
model_df = pd.read_csv(matches[0])

pred_cols = ['dist_roads','dist_parks','dist_coca','dist_mosaic',
             'temp_C','vpd_kPa','ndvi','wind_ms','oni']

# Spatial blocks (same 0.25 deg definition as v1 — needed everywhere GroupKFold is used)
BLOCK = 0.25
model_df['block'] = (model_df['lon']//BLOCK).astype(int).astype(str) + '_' + \
                    (model_df['lat']//BLOCK).astype(int).astype(str)

print("Dataset:", model_df.shape, "| blocks:", model_df['block'].nunique())
print("Year range:", model_df['year'].min(), "-", model_df['year'].max())


Loading: ../../data/model_dataset/model_dataset.csv
Dataset: (6231, 14) | blocks: 120
Year range: 2001 - 2024


In [2]:
# === MEJORA v2 vs v1: el GridSearchCV solo ve datos year <= 2019 ===
# ¿Por qué? En v1, GridSearchCV se entrenaba con TODO el dataset (2001-2024). Como los
# folds de GroupKFold se arman por bloque espacial (no por año), filas de 2020+
# terminaban como datos de ENTRENAMIENTO en algunos folds durante la búsqueda de
# hiperparámetros -> fuga temporal indirecta: los "mejores" hiperparámetros pudieron
# beneficiarse de ver patrones de los años que se suponía iban a quedar como hold-out
# temporal genuino.
#
# ¿Cómo se implementa el tuning? Se recorta el dataset a year <= 2019 ANTES de pasarlo
# a GridSearchCV. Los años >= 2020 quedan completamente afuera de esta celda y solo se
# usan una vez, más abajo, como hold-out temporal real. La grilla de hiperparámetros es
# IDÉNTICA a v1 (mismos valores candidatos) — el único cambio es el recorte de datos,
# para poder aislar el efecto de esta mejora específica.
tune_df = model_df[model_df['year'] <= 2019].copy()
print(f"Tuning subset: {tune_df.shape[0]} rows ({tune_df['block'].nunique()} blocks) "
      f"| years {tune_df['year'].min()}-{tune_df['year'].max()} "
      f"(year>=2020 rows are NOT included here)")

X_tune = tune_df[pred_cols].values
y_tune = tune_df['burned'].astype(int).values
groups_tune = tune_df['block'].values
cv = GroupKFold(n_splits=10)   # same 10-fold spatial-block CV as v1

# ---------- Logistic Regression grid (identical to v1 — only the DATA changed) ----------
pipeline_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=5000)),
])
param_grid_lr = {
    'clf__C': [0.01, 0.1, 1.0, 10.0],
    'clf__penalty': ['l1', 'l2'],
    'clf__solver': ['liblinear'],   # only solver here that supports both l1 and l2
}
# 4 x 2 x 1 = 8 candidates x 10 folds = 80 fits, scored with PR-AUC
lr_search = GridSearchCV(pipeline_lr, param_grid_lr, cv=cv,
                          scoring='average_precision', n_jobs=-1)
lr_search.fit(X_tune, y_tune, groups=groups_tune)   # <= 2019 ONLY
lr_search_best_parameters = lr_search.best_params_

# ---------- Random Forest grid (identical to v1 — only the DATA changed) ----------
param_grid_rf = {
    'n_estimators':     [200, 300, 500],
    'max_features':     ['sqrt', 0.5],
    'min_samples_leaf': [3, 5, 10, 20],
    'max_depth':        [None, 10, 20],
}
# 3 x 2 x 4 x 3 = 72 candidates x 10 folds = 720 fits, scored with PR-AUC
rf_search = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_grid_rf,
    cv=cv,
    scoring='average_precision',
    n_jobs=-1,
)
rf_search.fit(X_tune, y_tune, groups=groups_tune)   # <= 2019 ONLY
rf_search_best_parameters = rf_search.best_params_

n_candidates_lr = 4 * 2 * 1
n_candidates_rf = 3 * 2 * 4 * 3
print(f"LR grid: {n_candidates_lr} candidates x 10 folds = {n_candidates_lr * 10} fits (data: year<=2019 only)")
print(f"RF grid: {n_candidates_rf} candidates x 10 folds = {n_candidates_rf * 10} fits (data: year<=2019 only)")
print("Best LR hyperparameters (v2):", lr_search_best_parameters)
print("Best RF hyperparameters (v2):", rf_search_best_parameters)


Tuning subset: 5153 rows (119 blocks) | years 2001-2019 (year>=2020 rows are NOT included here)


c:\Users\Natal\.conda\envs\fire_thesis\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\Natal\.conda\envs\fire_thesis\Lib\site-packages\sklearn\linear_model\_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


LR grid: 8 candidates x 10 folds = 80 fits (data: year<=2019 only)
RF grid: 72 candidates x 10 folds = 720 fits (data: year<=2019 only)
Best LR hyperparameters (v2): {'clf__C': 0.01, 'clf__penalty': 'l1', 'clf__solver': 'liblinear'}
Best RF hyperparameters (v2): {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 5, 'n_estimators': 300}


In [3]:
# Tuned models — best configs found by GridSearchCV using ONLY year<=2019 data
best_lr = lr_search.best_estimator_
best_rf = rf_search.best_estimator_
print("Tuned LR (v2):", lr_search_best_parameters)
print("Tuned RF (v2):", rf_search_best_parameters)


Tuned LR (v2): {'clf__C': 0.01, 'clf__penalty': 'l1', 'clf__solver': 'liblinear'}
Tuned RF (v2): {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 5, 'n_estimators': 300}


In [4]:
def evaluate_model(name, estimator, model_df, pred_cols, n_splits=10, block=0.25):
    """
    Same validation protocol as v1 (spatial block CV of 10 folds over the FULL
    dataset + a single temporal hold-out train<=2019/test>=2020), PLUS confusion
    matrices, which v1 did not report:
      - spatial_cm: the 2x2 confusion matrix summed across the 10 spatial folds.
      - temporal_cm: the 2x2 confusion matrix from the single temporal hold-out.
    This function evaluates the FINAL model on the full dataset — it is separate
    from the hyperparameter search above (which used year<=2019 only). Reusing
    all years here is correct: spatial block CV never needed the temporal cutoff,
    and the temporal hold-out below re-fits the model from scratch on <=2019 only.
    """
    X = model_df[pred_cols].values
    y = model_df['burned'].astype(int).values

    blocks = (model_df['lon']//block).astype(int).astype(str) + '_' + \
             (model_df['lat']//block).astype(int).astype(str)
    groups = blocks.values

    # ---------- SPATIAL BLOCK CV ----------
    gkf = GroupKFold(n_splits=n_splits)
    rows = []
    cm_spatial = np.zeros((2, 2), dtype=int)
    for fold, (tr, te) in enumerate(gkf.split(X, y, groups), 1):
        m = clone(estimator).fit(X[tr], y[tr])          # fresh copy, refit per fold
        prob = m.predict_proba(X[te])[:, 1]
        pred = m.predict(X[te])
        auc   = roc_auc_score(y[te], prob)
        prauc = average_precision_score(y[te], prob)
        f1    = f1_score(y[te], pred)
        rows.append((auc, prauc, f1))
        cm_spatial += confusion_matrix(y[te], pred, labels=[0, 1])
        print(f"  Fold {fold:2d}: AUC={auc:.3f}  PR-AUC={prauc:.3f}  F1={f1:.3f}")

    r = np.array(rows)
    print(f"\nSPATIAL BLOCK CV — {name}")
    print(f"  AUC-ROC : {r[:,0].mean():.3f} ± {r[:,0].std():.3f}")
    print(f"  PR-AUC  : {r[:,1].mean():.3f} ± {r[:,1].std():.3f}")
    print(f"  F1      : {r[:,2].mean():.3f} ± {r[:,2].std():.3f}")
    print(f"  Confusion matrix (summed over 10 folds) [rows=true, cols=pred]:\n{cm_spatial}")

    # ---------- TEMPORAL SPLIT ----------
    tr = (model_df['year'] <= 2019).values
    te = (model_df['year'] >= 2020).values
    m = clone(estimator).fit(X[tr], y[tr])
    prob = m.predict_proba(X[te])[:, 1]
    pred = m.predict(X[te])
    cm_temporal = confusion_matrix(y[te], pred, labels=[0, 1])

    print(f"\nTEMPORAL SPLIT — {name}")
    print(f"  train ≤2019: {tr.sum()} rows ({int(y[tr].sum())} events) | "
          f"test ≥2020: {te.sum()} rows ({int(y[te].sum())} events)")
    print(f"  AUC-ROC : {roc_auc_score(y[te], prob):.3f}")
    print(f"  PR-AUC  : {average_precision_score(y[te], prob):.3f}")
    print(f"  F1      : {f1_score(y[te], pred):.3f}")
    print(f"  Confusion matrix (temporal hold-out) [rows=true, cols=pred]:\n{cm_temporal}")

    return {
        'spatial': r.mean(axis=0), 'spatial_std': r.std(axis=0),
        'spatial_cm': cm_spatial,
        'temporal': (roc_auc_score(y[te], prob),
                     average_precision_score(y[te], prob),
                     f1_score(y[te], pred)),
        'temporal_cm': cm_temporal,
    }


In [5]:
# Evaluate the v2-tuned models (hyperparameters chosen WITHOUT ever seeing year>=2020)
res_lr_v2 = evaluate_model("Logistic Regression (v2 tuned)", best_lr, model_df, pred_cols)

res_rf_v2 = evaluate_model("Random Forest (v2 tuned)", best_rf, model_df, pred_cols)


c:\Users\Natal\.conda\envs\fire_thesis\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\Natal\.conda\envs\fire_thesis\Lib\site-packages\sklearn\linear_model\_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
c:\Users\Natal\.conda\envs\fire_thesis\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ra

  Fold  1: AUC=0.900  PR-AUC=0.842  F1=0.809
  Fold  2: AUC=0.668  PR-AUC=0.497  F1=0.417
  Fold  3: AUC=0.847  PR-AUC=0.758  F1=0.706
  Fold  4: AUC=0.856  PR-AUC=0.789  F1=0.683
  Fold  5: AUC=0.876  PR-AUC=0.701  F1=0.512
  Fold  6: AUC=0.839  PR-AUC=0.709  F1=0.529
  Fold  7: AUC=0.830  PR-AUC=0.667  F1=0.645


c:\Users\Natal\.conda\envs\fire_thesis\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\Natal\.conda\envs\fire_thesis\Lib\site-packages\sklearn\linear_model\_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
c:\Users\Natal\.conda\envs\fire_thesis\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ra

  Fold  8: AUC=0.864  PR-AUC=0.796  F1=0.698
  Fold  9: AUC=0.769  PR-AUC=0.540  F1=0.444
  Fold 10: AUC=0.873  PR-AUC=0.729  F1=0.689

SPATIAL BLOCK CV — Logistic Regression (v2 tuned)
  AUC-ROC : 0.832 ± 0.064
  PR-AUC  : 0.703 ± 0.105
  F1      : 0.613 ± 0.123
  Confusion matrix (summed over 10 folds) [rows=true, cols=pred]:
[[3656  498]
 [ 891 1186]]

TEMPORAL SPLIT — Logistic Regression (v2 tuned)
  train ≤2019: 5153 rows (1828 events) | test ≥2020: 1078 rows (249 events)
  AUC-ROC : 0.808
  PR-AUC  : 0.617
  F1      : 0.557
  Confusion matrix (temporal hold-out) [rows=true, cols=pred]:
[[630 199]
 [ 76 173]]
  Fold  1: AUC=0.918  PR-AUC=0.875  F1=0.824
  Fold  2: AUC=0.773  PR-AUC=0.638  F1=0.592
  Fold  3: AUC=0.865  PR-AUC=0.777  F1=0.704
  Fold  4: AUC=0.880  PR-AUC=0.817  F1=0.733
  Fold  5: AUC=0.913  PR-AUC=0.770  F1=0.582
  Fold  6: AUC=0.885  PR-AUC=0.806  F1=0.665
  Fold  7: AUC=0.897  PR-AUC=0.769  F1=0.673
  Fold  8: AUC=0.882  PR-AUC=0.814  F1=0.690
  Fold  9: AUC=0.8

In [6]:
# === Guardar resultados oficiales de v2 ===
# Cómo se guardan: se arma una fila por modelo (LR, RF) con TODAS las métricas
# (AUC-ROC, PR-AUC, F1 -- espacial mean/std y temporal) más las 4 celdas de cada
# matriz de confusión (tn, fp, fn, tp), y se exporta a outputs/metrics/, siguiendo
# el "próximo paso sugerido" ya documentado en outputs/metrics/README.md (una tabla
# de métricas en CSV por versión de modelo, en vez de solo texto impreso).
def cm_to_dict(cm, prefix):
    tn, fp, fn, tp = cm.ravel()
    return {f'{prefix}_tn': int(tn), f'{prefix}_fp': int(fp),
            f'{prefix}_fn': int(fn), f'{prefix}_tp': int(tp)}

rows = []
for model_name, res, best_params in [
    ('Logistic Regression', res_lr_v2, lr_search_best_parameters),
    ('Random Forest', res_rf_v2, rf_search_best_parameters),
]:
    row = {
        'model': model_name,
        'best_params': str(best_params),
        'auc_spatial_mean': res['spatial'][0], 'auc_spatial_std': res['spatial_std'][0],
        'prauc_spatial_mean': res['spatial'][1], 'prauc_spatial_std': res['spatial_std'][1],
        'f1_spatial_mean': res['spatial'][2], 'f1_spatial_std': res['spatial_std'][2],
        'auc_temporal': res['temporal'][0],
        'prauc_temporal': res['temporal'][1],
        'f1_temporal': res['temporal'][2],
    }
    row.update(cm_to_dict(res['spatial_cm'], 'cm_spatial'))
    row.update(cm_to_dict(res['temporal_cm'], 'cm_temporal'))
    rows.append(row)

results_v2_df = pd.DataFrame(rows)
out_path = '../../outputs/metrics/tuning_v2_metrics.csv'   # repo-wide metrics folder
os.makedirs(os.path.dirname(out_path), exist_ok=True)
results_v2_df.to_csv(out_path, index=False)
print(f"Resultados guardados en: outputs/metrics/tuning_v2_metrics.csv\n")
results_v2_df


Resultados guardados en: outputs/metrics/tuning_v2_metrics.csv



,model,best_params,auc_spatial_mean,auc_spatial_std,prauc_spatial_mean,prauc_spatial_std,f1_spatial_mean,f1_spatial_std,auc_temporal,prauc_temporal,f1_temporal,cm_spatial_tn,cm_spatial_fp,cm_spatial_fn,cm_spatial_tp,cm_temporal_tn,cm_temporal_fp,cm_temporal_fn,cm_temporal_tp
0,Logistic Regression,"{'clf__C': 0.01, 'clf__penalty': 'l1', 'clf__s...",0.832048,0.064047,0.702705,0.104526,0.61330,0.122546,0.808198,0.616814,0.557166,3656,498,891,1186,630,199,76,173
1,Random Forest,"{'max_depth': None, 'max_features': 'sqrt', 'm...",0.876874,0.041418,0.779414,0.067380,0.68045,0.073988,0.819263,0.568723,0.560000,3736,418,762,1315,628,201,74,175
